# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR\(^2\) [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset Croissant schema is available at: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the FAIR\(^2\) dataset metadata and prepare for analysis using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Let us examine the available record sets, fields, and their `@id`s for reference in loading and processing. This helps us know which tables (`RecordSet`), columns/fields, and their identifiers are present in the dataset.

We will enumerate all `record_set`s, the fields within them (with their `@id`s), and display a preview of a few records.

In [ ]:
# Discover record sets and their field ids
record_sets = list(dataset.record_sets)
print(f"Record sets found ({len(record_sets)}):")
for rs in record_sets:
    print(f"- Record set name: {rs.name!r}")
    print(f"  @id: {rs.id!r}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name!r} (@id: {field.id}) [type: {field.data_type}]")
    print()

# If the dataset does not populate record_sets, try to infer from metadata (for exploratory use)
if len(record_sets) == 0:
    print("No record sets found in the Croissant model. Please verify the dataset model or inspect the schema for record sets.")

Below we attempt to preview records from any discovered record sets.

**Note:** This may produce results only if the dataset schema conforms to Croissant and provides actual data files/links for record sets. All references are made by `@id` for reproducibility.

In [ ]:
# If we have record sets, show a sample of their records
for rs in record_sets:
    print(f"Preview for Record Set: {rs.name} (@id: {rs.id})")
    try:
        rec_iter = dataset.records(record_set=rs.id)
        for i, rec in enumerate(rec_iter):
            print(f"Record {i}:\n{pprint.pformat(rec)}\n")
            if i >= 2:
                break
    except Exception as e:
        print(f"Could not read records for {rs.id}. Reason: {str(e)}")
    print("------------------\n")

## 3. Data Extraction
In this section, we extract data from the available record sets, referencing them by their `@id`. Each loaded record set is saved as a Pandas DataFrame for fast processing. 

We list out all relevant record set IDs, then load each to a DataFrame using `dataset.records(record_set=...)`. If there is only one record set, only that will be loaded and displayed.

In [ ]:
# Prepare to load all available record sets using their @id
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading data for Record Set: {rs_id}")
    try:
        df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
        dataframes[rs_id] = df
        print(f"Columns: {df.columns.tolist()}")
        display(df.head(3))
    except Exception as e:
        print(f"Could not load record set '{rs_id}': {str(e)}")

## 4. Exploratory Data Analysis (EDA)
Now that we have data loaded as a DataFrame, let's select a numeric field (e.g., peptide counts, molecular weight, normalized abundance) using its `@id`, filter records, normalize, and group by categorical variables (e.g., by experimental condition).

First, we enumerate numeric (and a possible group) fields from the schema for use by their `@id`.

In [ ]:
# Select record set and field IDs for analysis
if len(dataframes) == 0:
    print("No dataframes loaded. Please verify the Croissant schema and data availability.")
else:
    # Use the first available record set
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]

    # Discover numeric fields (from schema)
    numeric_fields = []
    group_fields = []
    for field in [f for r in record_sets if r.id == rs_id for f in r.fields]:
        if field.data_type in ['Integer', 'Float', 'Number']:
            numeric_fields.append(field.id)
        else:
            group_fields.append(field.id)

    print(f"Numeric fields: {numeric_fields}")
    print(f"Categorical/group fields: {group_fields}")

    # If available, pick the first numeric field for demo
    if len(numeric_fields) > 0:
        numeric_field = numeric_fields[0]  # use @id for the field
    else:
        print("No numeric fields detected. Please check the dataset.")

    # If available, pick first group field
    group_field = group_fields[0] if len(group_fields) > 0 else None

    # Filtering out missing/invalid data for the field
    if numeric_field in df:
        # Coerce to numeric in case columns are string-encoded
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].quantile(0.5) if df[numeric_field].notnull().sum() > 0 else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:0.3f} (median):\n")
        display(filtered_df[[numeric_field]].head(5))

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"].head()])
    else:
        print(f"Field {numeric_field} not present in dataframe columns.")

    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Let us visualize the distribution of the selected numeric field and a group summary if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped; check if dataframes and field names are correct.")

## 6. Conclusion
In this notebook, we demonstrated end-to-end data exploration and basic analysis of the FAIR\(^2\) Mass Spectrometry dataset, referencing all schema entities by their `@id`. With these reproducible steps, you can further extend processing, feature engineering, and modeling for your desired applications.

**Key points:**
- All references to record sets and fields are performed using their Croissant `@id`.
- The notebook dynamically inspects the structure, so you can adapt steps for other datasets described with Croissant schemas.
- Use this workflow as a template for FAIR dataset exploration with `mlcroissant`.